observations:
- depth sensor
- compass heading
- velocity

state: 

$$
x = 
\begin{bmatrix}
p_1 \\ p_2 \\ v_1 \\ v_2
\end{bmatrix}
$$

dynamics:

$$
F = 
\begin{bmatrix}
1 & 0 & \Delta t  & 0 \\
0 & 1 & 0 & \Delta t \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$


Observation function
$$
h(x) = 
\begin{bmatrix}
bath(p_1, p_2) \\ tan(\frac{p_2}{p_1}) \\ v_1 \\ v_2
\end{bmatrix}
$$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import ipywidgets as widgets
import seaborn as sns

from IPython.display import display
from scipy.interpolate import RegularGridInterpolator

sns.set_theme(context='notebook', style='darkgrid')

In [2]:
# multi-gaussian surface
def gaussian_surface(x_in, centers, A, covs):
    """
    x_in:    (N, 1, 2, 1)
    centers: (1, M, 2, 1)
    A:       (1, M)
    covs:    (1, M, 2, 2)
    """
    d = x_in - centers
    cov_inv = np.linalg.inv(covs)

    q = d.transpose(0, 1, 3, 2) @ cov_inv @ d
    q = q.squeeze((-1, -2))

    heights = A * np.exp(-0.5 * q)

    return heights.sum(axis=1)


def normal_pdf(x, mean=0.0, std=1.0):
    variance = std ** 2
    denominator = np.sqrt(2 * np.pi * variance)
    numerator = np.exp(-((x - mean) ** 2) / (2 * variance))
    return numerator / denominator
    

## Create Generate Simulated Bathymetric Map

In [3]:
nx, ny = (200, 200)

x = np.linspace(-1, 1, nx)
y = np.linspace(-1, 1, ny)

xv, yv = np.meshgrid(x, y)

# (N, 1, 2, 1)
coords = np.stack([xv.ravel(), yv.ravel()], axis=1)
coords = coords[:, None, :, None]

# Centers: (1, M, 2, 1)
centers = np.array([
    [[-0.3], [ 0.0]],
    [[ 0.5], [ 0.3]],
    [[ 0.0], [-0.6]],
])[None]

# Heights: (1, M)
# Positive = peak, negative = valley
A = np.array([[10.0, 7.0, -4.0]])

# One covariance matrix per peak/valley: (1, M, 2, 2)
covs = np.array([
    [[0.08, 0.00],
     [0.00, 0.04]],

    [[0.2, 0.00],
     [0.00, 0.1]],

    [[0.15, 0.00],
     [0.00, 0.05]],
])[None]

z = gaussian_surface(coords, centers, A, covs).reshape(ny, nx)

## Visualize Bathymetric Map

## Define Depth as Function of Position

In [10]:
get_depth = RegularGridInterpolator((x, y), z)
likelihood = lambda obs: normal_pdf(obs, z)

# Set heatmap axis labels
xtick_ixs = np.concat([np.arange(0, nx, 20), [nx - 1]])
xtick_labels = [f"{v:.2f}" for v in x[xtick_ixs]]

ytick_ixs = np.concat([np.arange(0, ny, 20), [ny - 1]])
ytick_labels = [f"{v:.2f}" for v in y[::-1][ytick_ixs]]

# Define a truncated color palette (10% less blacks)
base = plt.get_cmap("mako")
p = colors.LinearSegmentedColormap.from_list(
    "mako_truncated",
    base(np.linspace(0.1, 0.95, 256))  # truncate lower and upper color bounds
)

min_depth = z.min()
max_depth = z.max()
@widgets.interact(observation=widgets.FloatSlider(value=min_depth, min=min_depth, max=max_depth))
def likelihood_widget(observation):
    
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    
    sns.heatmap(z, cmap=p, ax=ax[0])
    sns.heatmap(likelihood(observation), ax=ax[1], xticklabels=False, yticklabels=False)

    ax[0].set_xticks(xtick_ixs, labels=xtick_labels, rotation=45)
    ax[0].set_yticks(ytick_ixs, labels=ytick_labels, rotation=0)
    ax[0].invert_yaxis()
    ax[1].invert_yaxis()

    ax[0].contour(
        z,
        levels=25,
        colors='k',
        linewidths=1,
        alpha=0.4
    )

interactive(children=(FloatSlider(value=-3.8833661844401037, description='observation', max=10.903878823665748…

In [ ]:
n_pixels = len(x) * len(y)

prior = np.zeros_like(z) + 1 / n_pixels
